Tutorial Part 4 – LLM-as-a-Judge Evaluation via Ollama

What you will learn:

   • How to use a LOCAL Ollama model as the evaluation judge
     (no OpenAI key required)

   • How to write a custom @scorer that calls the Ollama API
     directly to assess response quality

   • How to combine LLM-judge scorers with rule-based scorers


Judge scorers implemented here:
   ┌───────────────────┬──────────────────────────────────────┐
   │ Scorer            │ What the judge checks                │
   ├───────────────────┼──────────────────────────────────────┤
   │ llm_correctness   │ Is the answer factually correct?     │
   │ llm_helpfulness   │ Is the answer genuinely helpful?     │
   │ llm_language      │ Is the response written in English?  │
   └───────────────────┴──────────────────────────────────────┘

In [1]:
import os
import mlflow
import subprocess, sys

from mlflow.genai import scorer
from mlflow.entities import Feedback

from langchain_ollama import OllamaLLM, ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

1. Start the server

In [2]:
subprocess.Popen([
    sys.executable, "-m", "mlflow", "server",
    "--host", "127.0.0.1", "--port", "6001",
])

<Popen: returncode: None args: ['c:\\Users\\konan\\projects\\langchain-ollam...>

2. Setup

In [3]:
os.environ.setdefault("OPENAI_API_KEY", "dummy")

mlflow.set_tracking_uri("http://localhost:6001")
mlflow.set_experiment("langchain-ollama-llm-judge")
mlflow.langchain.autolog()

2026/10/25 10:47:30 INFO mlflow.tracking.fluent: Experiment with name 'langchain-ollama-llm-judge' does not exist. Creating a new experiment.


3. The model under test

In [9]:
llm = OllamaLLM(
    model="qwen3.6",
    base_url="http://localhost:11434",
    temperature=0.1,
)

prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Answer concisely.\n\nQuestion: {question}"
)
chain = prompt | llm

def qa_predict_fn(question: str) -> str:
    return chain.invoke({"question": question})

4. Judge model (same Ollama endpoint, can be a diff model)

In [10]:
judge_llm = ChatOllama(
    model="deepseek-r1:1.5b",
    base_url="http://localhost:11434",
    temperature=0.0,   # deterministic judgements
)

def _judge(system_prompt: str, user_prompt: str) -> str:
    """Call the local Ollama judge and return its raw text response."""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    response = judge_llm.invoke(messages)
    return response.content.strip()

5. LLM-judge custom scorers

In [11]:
@scorer
def llm_correctness(inputs: dict, outputs: str, expectations: dict) -> Feedback:
    """
    Ask the judge: is the answer factually correct given the expected answer?
    Returns a score between 0 and 1 and a brief rationale.
    """
    expected = expectations.get("expected_response", "")
    system = (
        "You are an evaluation judge. "
        "Respond with ONLY a JSON object: {\"score\": <0.0-1.0>, \"reason\": \"<one sentence>\"}"
    )
    user = (
        f"Question: {inputs.get('question', '')}\n"
        f"Expected answer: {expected}\n"
        f"Actual answer: {outputs}\n\n"
        "Score 1.0 if the actual answer is fully correct, 0.5 if partially correct, 0.0 if wrong."
    )
    raw = _judge(system, user)
    import json, re
    try:
        # Extract JSON even if surrounded by markdown fences
        match = re.search(r"\{.*?\}", raw, re.DOTALL)
        data = json.loads(match.group()) if match else {}
        return Feedback(name="llm_correctness", value=float(data.get("score", 0.0)), rationale=data.get("reason", raw))
    except Exception as e:
        return Feedback(name="llm_correctness", value=0.0, rationale=raw, error=str(e))


@scorer
def llm_helpfulness(inputs: dict, outputs: str) -> Feedback:
    """
    Ask the judge: is the response genuinely helpful to the user?
    Returns a score between 0 and 1.
    """
    system = (
        "You are an evaluation judge. "
        "Respond with ONLY a JSON object: {\"score\": <0.0-1.0>, \"reason\": \"<one sentence>\"}"
    )
    user = (
        f"Question: {inputs.get('question', '')}\n"
        f"Response: {outputs}\n\n"
        "Score 1.0 if fully helpful, 0.5 if somewhat helpful, 0.0 if unhelpful or evasive."
    )
    raw = _judge(system, user)
    import json, re
    try:
        match = re.search(r"\{.*?\}", raw, re.DOTALL)
        data = json.loads(match.group()) if match else {}
        return Feedback(name="llm_helpfulness", value=float(data.get("score", 0.0)), rationale=data.get("reason", raw))
    except Exception as e:
        return Feedback(name="llm_helpfulness", value=0.0, rationale=raw, error=str(e))


@scorer
def llm_language_check(outputs: str) -> bool:
    """Ask the judge: is the response written in English?"""
    system = "You are a language detector. Reply with only YES or NO."
    user = f"Is the following text written in English?\n\n{outputs}"
    raw = _judge(system, user)
    return "yes" in raw.lower()

6. Dataset

In [12]:
eval_dataset = [
    {
        "inputs":  {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"},
    },
    {
        "inputs":  {"question": "Who invented the telephone?"},
        "expectations": {"expected_response": "Alexander Graham Bell"},
    },
    {
        "inputs":  {"question": "What is the boiling point of water in Celsius?"},
        "expectations": {"expected_response": "100 degrees Celsius"},
    },
]

7. Run evaluation 

In [13]:
print("Starting LLM-judge evaluation (local Ollama judge) …")
print("Note: each scorer calls the judge LLM – this takes ~1-2 min\n")

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=qa_predict_fn,
    scorers=[llm_correctness, llm_helpfulness, llm_language_check],
)

print("\n" + "="*60)
print("LLM-JUDGE EVALUATION RESULTS")
print("="*60)
print("\nAggregate metrics:")
for k, v in results.metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")
print("\nPer-sample results:")
scorer_cols = [c for c in results.result_df.columns if c.endswith("/value")]
print(results.result_df[["request", "response"] + scorer_cols].to_string(index=False))
print("\n✅  Done – open http://localhost:6001 to inspect trace assessments.")

Starting LLM-judge evaluation (local Ollama judge) …
Note: each scorer calls the judge LLM – this takes ~1-2 min



2026/10/25 10:50:21 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/10/25 11:06:13 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'llm_correctness': 3/3 failed, 'llm_helpfulness': 3/3 failed, 'llm_language_check': 3/3 failed. Check individual trace assessments for detailed error messages.



LLM-JUDGE EVALUATION RESULTS

Aggregate metrics:

Per-sample results:
                                                       request               response  llm_helpfulness/value  llm_language_check/value  llm_correctness/value expected_response/value
                {'question': 'What is the capital of France?'}                 Paris.                    NaN                       NaN                    NaN                   Paris
                   {'question': 'Who invented the telephone?'} Alexander Graham Bell.                    NaN                       NaN                    NaN   Alexander Graham Bell
{'question': 'What is the boiling point of water in Celsius?'}                  100°C                    NaN                       NaN                    NaN     100 degrees Celsius

✅  Done – open http://localhost:6001 to inspect trace assessments.


[Trace(trace_id='tr-214cb03b6ccd2c4a6e1305d0a98332d7'), Trace(trace_id='tr-b98e0415e3a3fc991d2cf06bb76b293e'), Trace(trace_id='tr-5044f9816735d256db0d83b0315ba0b1')]